# Session 12: Applications and projects — comparison with traditional methods and capstone project

Welcome to the final session of the course. Sessions 1–11 have built the full machinery of physics-informed neural networks (PINNs): from differential equations and finite differences ([Sessions 1–2](Session1.ipynb)), through neural network theory and automatic differentiation ([Sessions 3–4](Session3.ipynb)), the core PINN framework ([Sessions 5–6](Session5.ipynb)), advanced partial differential equations (PDEs) and implementation ([Sessions 7–8](Session7.ipynb)), tuning and adaptive strategies ([Sessions 9–10](Session9.ipynb)), and physical applications including the Schrödinger equation and Navier–Stokes equations ([Session 11](Session11.ipynb)).

This session has three parts:

1. **Part 1 — Systematic comparison: PINNs vs finite differences.** A head-to-head benchmark on the 1D heat equation from [Session 6](Session6.ipynb), with controlled variation of resolution and network size.
2. **Part 2 — Open project: damped pendulum.** A guided self-study project for the nonlinear pendulum ordinary differential equation (ODE), with a scaffold for you to complete and a worked solution hidden behind a collapsible block.
3. **Part 3 — Course summary.** A structured review of the 12-session arc, key strengths and limitations of PINNs, and pointers to further reading.

By the end of this session you will be able to make an informed, quantitative argument about when to choose a PINN over a classical numerical method, and when not to.

## 1. Setup

In [1]:
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg
import scipy.integrate
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Fix random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Problem constants (same as Session 6)
ALPHA = 0.01
PI    = np.pi

Using device: cpu


---

## 2. Part 1: systematic comparison of PINNs and finite differences on the 1D heat equation

### 2.1 Problem statement

We revisit the 1D heat equation studied in [Session 6](Session6.ipynb):

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2},
\quad x \in [0, 1],\; t \in [0, 1]
$$

with thermal diffusivity $\alpha = 0.01$, boundary conditions $u(0,t) = u(1,t) = 0$, and initial condition $u(x,0) = \sin(\pi x)$. The exact solution is:

$$
u(x, t) = e^{-\alpha \pi^2 t}\sin(\pi x)
$$

### 2.2 Metrics and comparison axes

For a fair comparison we record three quantities for each solver configuration:

| Metric | Symbol | Definition |
|---|---|---|
| Relative $L^2$ error | $\varepsilon$ | $\|u_{\text{approx}} - u_{\text{exact}}\|_2 / \|u_{\text{exact}}\|_2$ over a $100 \times 100$ evaluation grid |
| Wall-clock time | $T_{\text{wall}}$ | Training time (PINN) or solve time (FD) in seconds |
| Degrees of freedom | DOF | Number of grid points $N^2$ (FD) or number of trainable parameters (PINN) |

Plotting $\varepsilon$ against DOF on shared axes is the standard way to visualise computational efficiency.

### 2.3 Crank–Nicolson finite-difference (FD) solver

The Crank–Nicolson scheme, introduced in [Session 2](Session2.ipynb), is unconditionally stable and second-order accurate in both space and time. For $N$ internal spatial grid points and time step $\Delta t = 1/N_t$ (we set $N_t = N$ for simplicity so that DOF $= N^2$), the update at each time step solves a tridiagonal system:

$$
\left(\mathbf{I} - \frac{r}{2}\mathbf{D}_2\right) \mathbf{u}^{n+1}
= \left(\mathbf{I} + \frac{r}{2}\mathbf{D}_2\right) \mathbf{u}^{n}
$$

where $r = \alpha \Delta t / (\Delta x)^2$ and $\mathbf{D}_2$ is the second-difference matrix.

In [2]:
def solve_heat_crank_nicolson(N):
    """
    Solve u_t = alpha * u_xx on [0,1]x[0,1] using the Crank-Nicolson scheme.

    Parameters
    ----------
    N : int
        Number of internal spatial grid points.  Nt = N time steps are used,
        giving DOF = N^2.

    Returns
    -------
    u_final : ndarray, shape (N+2,)
        Solution at t=1, including boundary points.
    x_grid : ndarray, shape (N+2,)
        Spatial grid including boundary points.
    solve_time : float
        Wall-clock time in seconds.
    dof : int
        Total degrees of freedom N * N (space steps * time steps).
    """
    dx = 1.0 / (N + 1)
    Nt = N                         # equal number of time steps
    dt = 1.0 / Nt
    r  = ALPHA * dt / dx**2

    x_full = np.linspace(0.0, 1.0, N + 2)   # includes x=0 and x=1
    x_int  = x_full[1:-1]                   # internal points only

    # Initial condition on internal points
    u = np.sin(PI * x_int)

    # Tridiagonal matrices for Crank-Nicolson
    diag  = np.ones(N)
    offdiag = np.ones(N - 1)

    A_lhs = (np.diag(diag * (1.0 + r))
             - np.diag(offdiag * 0.5 * r, 1)
             - np.diag(offdiag * 0.5 * r, -1))   # LHS: (I - r/2 * D2)

    A_rhs = (np.diag(diag * (1.0 - r))
             + np.diag(offdiag * 0.5 * r, 1)
             + np.diag(offdiag * 0.5 * r, -1))   # RHS: (I + r/2 * D2)

    # Pre-factorise the LHS (re-used at every time step)
    lu, piv = scipy.linalg.lu_factor(A_lhs)

    t0 = time.perf_counter()
    for _ in range(Nt):
        rhs = A_rhs @ u
        u   = scipy.linalg.lu_solve((lu, piv), rhs)
    solve_time = time.perf_counter() - t0

    # Attach boundary values (always zero)
    u_full = np.concatenate([[0.0], u, [0.0]])
    return u_full, x_full, solve_time, N * Nt


def exact_solution(x, t, alpha=ALPHA):
    """Exact solution u = exp(-alpha*pi^2*t) * sin(pi*x)."""
    return np.exp(-alpha * PI**2 * t) * np.sin(PI * x)


def relative_l2_error_fd(N):
    """
    Compute the relative L2 error of the Crank-Nicolson solution at t=1
    against the exact solution on the solver's own spatial grid.
    """
    u_num, x_grid, solve_time, dof = solve_heat_crank_nicolson(N)
    u_ex  = exact_solution(x_grid, t=1.0)
    error = np.sqrt(np.mean((u_num - u_ex)**2)) / np.sqrt(np.mean(u_ex**2))
    return error, solve_time, dof


# Quick sanity check
err_check, t_check, dof_check = relative_l2_error_fd(20)
print(f'Crank-Nicolson N=20: error={err_check:.3e}, time={t_check:.4f}s, DOF={dof_check}')

Crank-Nicolson N=20: error=1.837e-04, time=0.0002s, DOF=400


### 2.4 PINN solver

The PINN architecture and loss function are identical to those in [Session 6](Session6.ipynb). We vary only the network width $W \in \{16, 32, 64, 128\}$, keeping the depth fixed at four hidden layers and the number of training epochs fixed at 8 000. This isolates the effect of the number of trainable parameters (DOF) on accuracy.

In [3]:
class HeatPINN(nn.Module):
    """MLP with tanh activations. Input: (x, t), output: u.

    Parameters
    ----------
    width : int
        Number of neurons in each of the four hidden layers.
    """
    def __init__(self, width=32):
        super().__init__()
        layers = [2, width, width, width, width, 1]
        seq = []
        for i in range(len(layers) - 2):
            seq += [nn.Linear(layers[i], layers[i + 1]), nn.Tanh()]
        seq.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*seq)

    def forward(self, X):
        return self.net(X)


def count_parameters(model):
    """Return the total number of trainable parameters in a PyTorch model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_heat_pinn(width, epochs=8000, n_colloc=10000, n_bc=200, n_ic=200,
                   lr=1e-3, seed=42):
    """
    Train a PINN for the 1D heat equation and return accuracy metrics.

    Returns
    -------
    rel_l2 : float
        Relative L2 error on a 100x100 evaluation grid.
    train_time : float
        Wall-clock training time in seconds.
    dof : int
        Number of trainable parameters.
    """
    torch.manual_seed(seed)
    model = HeatPINN(width=width).to(device)
    dof   = count_parameters(model)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    # --- Sampling ---
    colloc = torch.rand(n_colloc, 2, requires_grad=True, device=device)

    t_bc     = torch.rand(n_bc, device=device)
    bc_left  = torch.stack([torch.zeros(n_bc, device=device), t_bc], dim=1)
    bc_right = torch.stack([torch.ones(n_bc, device=device),  t_bc], dim=1)
    bc = torch.cat([bc_left, bc_right], dim=0)

    x_ic = torch.rand(n_ic, device=device)
    ic   = torch.stack([x_ic, torch.zeros(n_ic, device=device)], dim=1)
    u_ic_true = torch.sin(PI * x_ic).reshape(-1, 1)

    # --- Training loop ---
    t0 = time.perf_counter()
    for epoch in range(epochs):
        opt.zero_grad()

        # PDE residual: u_t - alpha * u_xx = 0
        u_c  = model(colloc)
        grad = torch.autograd.grad(
            u_c, colloc,
            grad_outputs=torch.ones_like(u_c),
            create_graph=True, retain_graph=True
        )[0]
        u_t  = grad[:, 1:2]
        u_x  = grad[:, 0:1]
        u_xx = torch.autograd.grad(
            u_x, colloc,
            grad_outputs=torch.ones_like(u_x),
            create_graph=True, retain_graph=True
        )[0][:, 0:1]

        loss_pde = torch.mean((u_t - ALPHA * u_xx)**2)
        loss_bc  = torch.mean(model(bc)**2)
        loss_ic  = torch.mean((model(ic) - u_ic_true)**2)

        loss = loss_pde + loss_bc + loss_ic
        loss.backward()
        opt.step()

    train_time = time.perf_counter() - t0

    # --- Evaluation on 100x100 grid ---
    with torch.no_grad():
        Ng = 100
        x_g = torch.linspace(0, 1, Ng)
        t_g = torch.linspace(0, 1, Ng)
        Xg, Tg = torch.meshgrid(x_g, t_g, indexing='ij')
        XT = torch.stack([Xg.flatten(), Tg.flatten()], dim=1).to(device)
        u_pred = model(XT).cpu().numpy().reshape(Ng, Ng)

    u_ex = exact_solution(Xg.numpy(), Tg.numpy())
    rel_l2 = (np.sqrt(np.mean((u_pred - u_ex)**2))
              / np.sqrt(np.mean(u_ex**2)))

    print(f'  Width={width:3d} | DOF={dof:6d} | '
          f'Error={rel_l2:.3e} | Time={train_time:.1f}s')
    return rel_l2, train_time, dof

### 2.5 Running the benchmark

The cells below run both solvers across their respective parameter ranges. On a typical laptop CPU the PINN runs take roughly 1–3 minutes each; total wall-clock time for the full benchmark is around 15–20 minutes. You may reduce `epochs` for a quicker but less accurate PINN result.

In [4]:
# --- Finite-difference benchmark ---
fd_grid_sizes = [10, 20, 50, 100]
fd_results = {}  # N -> (error, time, dof)

print('=== Crank-Nicolson finite differences ===')
for N in fd_grid_sizes:
    err, t_wall, dof = relative_l2_error_fd(N)
    fd_results[N] = (err, t_wall, dof)
    print(f'  N={N:3d} | DOF={dof:6d} | Error={err:.3e} | Time={t_wall:.6f}s')

=== Crank-Nicolson finite differences ===
  N= 10 | DOF=   100 | Error=6.685e-04 | Time=0.000185s
  N= 20 | DOF=   400 | Error=1.837e-04 | Time=0.000137s
  N= 50 | DOF=  2500 | Error=3.117e-05 | Time=0.000519s
  N=100 | DOF= 10000 | Error=7.949e-06 | Time=0.001048s


In [ ]:
# --- PINN benchmark ---
pinn_widths = [16, 32, 64, 128]
pinn_results = {}  # width -> (error, time, dof)

print('=== PINN (4 hidden layers, 8000 epochs) ===')
for W in pinn_widths:
    err, t_wall, dof = train_heat_pinn(width=W, epochs=8000)
    pinn_results[W] = (err, t_wall, dof)

=== PINN (4 hidden layers, 8000 epochs) ===
  Width= 16 | DOF=   881 | Error=6.048e-04 | Time=35.9s


### 2.6 Results: error vs degrees of freedom

In [ ]:
fd_dofs   = [fd_results[N][2]  for N in fd_grid_sizes]
fd_errors = [fd_results[N][0]  for N in fd_grid_sizes]
fd_times  = [fd_results[N][1]  for N in fd_grid_sizes]

pinn_dofs   = [pinn_results[W][2]  for W in pinn_widths]
pinn_errors = [pinn_results[W][0]  for W in pinn_widths]
pinn_times  = [pinn_results[W][1]  for W in pinn_widths]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel 1: error vs DOF ---
ax = axes[0]
ax.loglog(fd_dofs, fd_errors, 'o-', lw=2, ms=8, color='royalblue',
          label='Crank-Nicolson FD')
ax.loglog(pinn_dofs, pinn_errors, 's--', lw=2, ms=8, color='tomato',
          label='PINN')
for N, dof, err in zip(fd_grid_sizes, fd_dofs, fd_errors):
    ax.annotate(f'N={N}', (dof, err), textcoords='offset points',
                xytext=(6, 4), fontsize=8, color='royalblue')
for W, dof, err in zip(pinn_widths, pinn_dofs, pinn_errors):
    ax.annotate(f'W={W}', (dof, err), textcoords='offset points',
                xytext=(6, -12), fontsize=8, color='tomato')
ax.set_xlabel('Degrees of freedom (DOF)')
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('Accuracy vs DOF')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# --- Panel 2: error vs wall-clock time ---
ax = axes[1]
ax.loglog(fd_times, fd_errors, 'o-', lw=2, ms=8, color='royalblue',
          label='Crank-Nicolson FD')
ax.loglog(pinn_times, pinn_errors, 's--', lw=2, ms=8, color='tomato',
          label='PINN')
ax.set_xlabel('Wall-clock time (s)')
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('Accuracy vs wall-clock time')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.suptitle('PINNs vs Crank-Nicolson FD — 1D heat equation benchmark',
             fontsize=13)
plt.tight_layout()
plt.show()

# --- Summary table ---
print('\nFinite-difference results:')
print(f'{"N":>6} {"DOF":>8} {"Error":>12} {"Time (s)":>12}')
for N in fd_grid_sizes:
    err, tw, dof = fd_results[N]
    print(f'{N:6d} {dof:8d} {err:12.3e} {tw:12.6f}')

print('\nPINN results:')
print(f'{"Width":>6} {"DOF":>8} {"Error":>12} {"Time (s)":>12}')
for W in pinn_widths:
    err, tw, dof = pinn_results[W]
    print(f'{W:6d} {dof:8d} {err:12.3e} {tw:12.2f}')

### 2.7 Discussion

The benchmark above reveals a clear empirical picture. Consider the following points as you inspect your own results.

**When does FD win?**

- **Smooth, low-dimensional problems with known geometry.** The heat equation on a uniform 1D grid is the ideal FD problem. Crank–Nicolson achieves very small errors (often below $10^{-4}$) with only $N=100$ grid points and negligible wall-clock time (milliseconds). No amount of PINN hyperparameter tuning will match this on pure accuracy-per-second.
- **Predictable convergence rates.** FD methods converge at known algebraic rates ($O(h^2)$ for Crank–Nicolson in both space and time). There are no stochastic training dynamics, no risk of the optimiser getting stuck in a poor local minimum, and no loss-balancing decisions to make.
- **Large, time-dependent problems.** If you need the full space-time trajectory $u(x,t)$ for all $t$, a time-marching scheme computes it incrementally. The PINN solves for the entire domain simultaneously, which requires more memory and compute.

**When does PINN win (or at least, become competitive)?**

- **High-dimensional domains.** FD requires a grid: $10^d$ points for $d$ dimensions. A PINN samples the domain randomly, avoiding the curse of dimensionality. For $d \geq 4$, PINNs can be the only tractable mesh-free option.
- **Irregular geometries.** Building a structured FD grid for a domain shaped like, for example, the cross-section of a bone or a fluid-filled ventricle requires significant effort. A PINN needs only a method to sample interior points and detect boundary membership — often much simpler to implement.
- **Inverse problems.** As demonstrated in [Session 8](Session8.ipynb), embedding an unknown parameter as a learnable `nn.Parameter` is trivial in the PINN framework. The equivalent FD approach requires a separate outer optimisation loop.
- **Partial or scattered data.** When experimental measurements exist on an irregular grid (or with gaps), a PINN naturally combines the data loss and PDE loss in a single objective. FD methods do not accommodate scattered data straightforwardly.
- **Continuous representation.** A trained PINN is a smooth, differentiable function that can be evaluated at any $(x,t)$ without interpolation. This is useful in design optimisation or when the solution is needed at query points not known in advance.

**Summary rule of thumb.** For a well-posed forward problem on a structured domain in up to three spatial dimensions, use a classical solver. Turn to PINNs when the problem is high-dimensional, the domain is complex, unknown parameters must be inferred from data, or a continuous differentiable representation is needed.

---

## 3. Part 2: open project — the nonlinear pendulum

### 3.1 Problem statement

The simple pendulum of length $L$ in a gravitational field $g$ satisfies:

$$
\frac{d^2\theta}{dt^2} + \frac{g}{L}\sin\theta = 0
$$

with $g = 9.81\,\mathrm{m\,s^{-2}}$, $L = 1.0\,\mathrm{m}$, initial angle $\theta(0) = \pi/3$ (60°), and initial angular velocity $\dot{\theta}(0) = 0$.

This is a **nonlinear ODE** — the $\sin\theta$ term prevents a closed-form solution in general. The standard small-angle approximation $\sin\theta \approx \theta$ is inaccurate at 60°. We therefore use `scipy.integrate.solve_ivp` to generate a high-accuracy numerical reference solution.

The nonlinearity is handled by the PINN in exactly the same way as the Burgers' equation in [Session 7](Session7.ipynb): autograd computes $d^2\theta/dt^2$ from the network output and the residual is evaluated at collocation points.

### 3.2 Reference solution via `solve_ivp`

In [ ]:
# Physical parameters
G_GRAV = 9.81   # gravitational acceleration, m/s^2
L_PEND = 1.0    # pendulum length, m
THETA0 = PI / 3 # initial angle, radians (60 degrees)
DTHETA0 = 0.0   # initial angular velocity, rad/s
T_END = 6.0     # solve for 6 seconds (approximately two full periods)


def pendulum_ode(t, y):
    """Right-hand side of the pendulum system.

    State vector: y = [theta, dtheta/dt].
    Returns dy/dt = [dtheta/dt, -(g/L)*sin(theta)].
    """
    theta, omega = y
    return [omega, -(G_GRAV / L_PEND) * np.sin(theta)]


# Generate the reference solution using a high-order Runge-Kutta integrator
t_eval_ref = np.linspace(0, T_END, 600)
ref_sol = scipy.integrate.solve_ivp(
    pendulum_ode,
    t_span=(0.0, T_END),
    y0=[THETA0, DTHETA0],
    method='RK45',
    t_eval=t_eval_ref,
    rtol=1e-10,
    atol=1e-12,
)

t_ref    = ref_sol.t
theta_ref = ref_sol.y[0]   # angle theta(t)
omega_ref = ref_sol.y[1]   # angular velocity

print(f'Reference solution: {len(t_ref)} time points over [0, {T_END}] s')
print(f'Maximum angle: {np.max(np.abs(theta_ref)):.4f} rad '
      f'({np.degrees(np.max(np.abs(theta_ref))):.1f} deg)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_ref, np.degrees(theta_ref), 'k-', lw=2)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Angle (degrees)')
axes[0].set_title('Pendulum angle $\\theta(t)$ — reference solution')
axes[0].axhline(0, color='grey', lw=0.7, ls='--')
axes[0].grid(alpha=0.3)

axes[1].plot(np.degrees(theta_ref), np.degrees(omega_ref), 'k-', lw=1.5)
axes[1].set_xlabel('Angle $\\theta$ (degrees)')
axes[1].set_ylabel('Angular velocity $\\dot{\\theta}$ (deg/s)')
axes[1].set_title('Phase portrait')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Your task: implement the pendulum PINN

Below is a scaffold with `TODO` comments marking the sections you need to complete. Read through the entire scaffold before writing any code — the structure mirrors the PINN implementations in [Sessions 5](Session5.ipynb)–[9](Session9.ipynb).

**Hints**

- The ODE is second-order, so you need $d^2\theta/dt^2$ via two nested `torch.autograd.grad` calls (see [Session 4](Session4.ipynb) for the pattern with `create_graph=True`).
- The IC has two components: $\theta(0) = \pi/3$ and $\dot{\theta}(0) = 0$. Both must appear in the loss.
- The time domain $[0, 6]$ s is longer than previous examples. Consider whether you need more collocation points than in [Session 9](Session9.ipynb).
- `torch.sin` is differentiable, so the nonlinear term is handled automatically by autograd.

In [ ]:
# ============================================================
# SCAFFOLD — fill in every section marked TODO
# ============================================================

import torch
import torch.nn as nn
import numpy as np


class PendulumPINN(nn.Module):
    """
    MLP that maps time t -> angle theta(t).

    Input dimension  : 1  (scalar time)
    Output dimension : 1  (scalar angle)

    TODO: Choose the number of hidden layers and neurons per layer.
          Hint: 3-4 hidden layers of width 32-64 work well here.
          Use tanh activations throughout.
    """
    def __init__(self, width=32):
        super().__init__()
        # TODO: define self.net as an nn.Sequential of Linear + Tanh layers
        # followed by a final Linear(width, 1) output layer.
        raise NotImplementedError('TODO: implement PendulumPINN.__init__')

    def forward(self, t):
        # TODO: pass t through self.net and return the result
        raise NotImplementedError('TODO: implement PendulumPINN.forward')


def pendulum_ode_residual(model, t_col, g=G_GRAV, L=L_PEND):
    """
    Compute the ODE residual  d^2theta/dt^2 + (g/L)*sin(theta)  at collocation
    points t_col.

    Parameters
    ----------
    model  : PendulumPINN — the network theta_theta(t)
    t_col  : torch.Tensor of shape (N, 1) with requires_grad=True
    g, L   : physical parameters

    Returns
    -------
    residual : torch.Tensor of shape (N, 1)
    """
    # TODO: evaluate the network at t_col to get theta
    # TODO: compute dtheta/dt using torch.autograd.grad (first derivative)
    # TODO: compute d^2theta/dt^2 (second derivative, needs create_graph=True
    #        on the first differentiation)
    # TODO: return  d2theta_dt2 + (g/L) * torch.sin(theta)
    raise NotImplementedError('TODO: implement pendulum_ode_residual')


def train_pendulum_pinn(width=32, n_colloc=3000, epochs=15000, lr=1e-3,
                        t_end=T_END, seed=0):
    """
    Train the pendulum PINN and return the trained model.

    Returns
    -------
    model : trained PendulumPINN
    loss_history : list of float
    """
    torch.manual_seed(seed)
    model = PendulumPINN(width=width).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    # --- Collocation points: sample t uniformly in (0, t_end) ---
    # TODO: create t_col as a tensor of shape (n_colloc, 1) with
    #       values in [0, t_end] and requires_grad=True

    # --- Initial condition points ---
    t_ic  = torch.zeros(1, 1, device=device)   # t = 0
    # theta(0) = pi/3
    theta_ic_true = torch.tensor([[THETA0]], dtype=torch.float32, device=device)

    history = []
    for epoch in range(epochs):
        opt.zero_grad()

        # ODE residual loss
        # TODO: call pendulum_ode_residual and compute its mean squared value

        # Initial angle loss: (theta(0) - pi/3)^2
        # TODO: evaluate model(t_ic) and compare to theta_ic_true

        # Initial velocity loss: (dtheta/dt|_{t=0} - 0)^2
        # TODO: differentiate the model output at t_ic to get dtheta/dt(0)
        #       and compute the MSE against 0

        # TODO: combine the three losses (consider weighting the IC terms)
        # loss = loss_ode + lambda_theta * loss_ic_theta + lambda_omega * loss_ic_omega

        # TODO: call loss.backward() and opt.step()

        # TODO: append loss.item() to history

        if epoch % 3000 == 0:
            print(f'  Epoch {epoch:6d} | Loss: {history[-1]:.3e}')

    return model, history


# Uncomment the lines below to run your implementation:
# pend_model, pend_history = train_pendulum_pinn(width=32, epochs=15000)
print('Scaffold loaded. Fill in the TODO sections above, then uncomment the last line.')

### 3.4 Evaluation scaffold

Once your implementation runs, use the cell below to evaluate and plot your PINN against the reference solution.

In [ ]:
# Run this cell only after pend_model has been trained
# (i.e. after you have completed the TODOs and uncommented the training call).

try:
    t_eval_tensor = torch.linspace(0, T_END, 600).reshape(-1, 1).to(device)
    with torch.no_grad():
        theta_pinn = pend_model(t_eval_tensor).cpu().numpy().flatten()

    t_eval_np = t_eval_tensor.cpu().numpy().flatten()

    rel_l2_pend = (np.sqrt(np.mean((theta_pinn - theta_ref)**2))
                   / np.sqrt(np.mean(theta_ref**2)))
    print(f'Relative L2 error vs reference: {rel_l2_pend:.4e}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(t_ref, np.degrees(theta_ref), 'k-', lw=2,
                 label='Reference (solve_ivp)')
    axes[0].plot(t_eval_np, np.degrees(theta_pinn), 'r--', lw=2,
                 label='PINN')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Angle (degrees)')
    axes[0].set_title('Pendulum PINN vs reference solution')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].semilogy(pend_history)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Total loss')
    axes[1].set_title('Training loss history')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

except NameError:
    print('pend_model is not yet defined — complete the scaffold above first.')

### 3.5 Worked solution

Attempt the implementation yourself before expanding the solution below.

<details>
<summary><strong>Click to reveal the worked solution</strong></summary>

```python
import torch
import torch.nn as nn
import numpy as np


class PendulumPINN(nn.Module):
    """MLP: scalar t -> scalar theta(t). Four hidden layers of width 32."""

    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,     width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1),
        )

    def forward(self, t):
        return self.net(t)


def pendulum_ode_residual(model, t_col, g=9.81, L=1.0):
    """ODE residual d^2theta/dt^2 + (g/L)*sin(theta) at t_col."""
    theta = model(t_col)                              # theta(t)

    # First derivative: dtheta/dt
    dtheta_dt = torch.autograd.grad(
        theta, t_col,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    # Second derivative: d^2theta/dt^2
    d2theta_dt2 = torch.autograd.grad(
        dtheta_dt, t_col,
        grad_outputs=torch.ones_like(dtheta_dt),
        create_graph=True, retain_graph=True
    )[0]

    return d2theta_dt2 + (g / L) * torch.sin(theta)


def train_pendulum_pinn(width=32, n_colloc=3000, epochs=15000, lr=1e-3,
                        t_end=6.0, seed=0):
    """Train the pendulum PINN."""
    torch.manual_seed(seed)
    model = PendulumPINN(width=width).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    # Collocation points sampled uniformly in [0, t_end]
    t_col = (torch.rand(n_colloc, 1, device=device) * t_end).requires_grad_(True)

    # Initial conditions
    t_ic  = torch.zeros(1, 1, device=device)
    theta_ic_true = torch.tensor([[THETA0]], dtype=torch.float32, device=device)

    history = []
    for epoch in range(epochs):
        opt.zero_grad()

        # ODE residual loss
        residual    = pendulum_ode_residual(model, t_col)
        loss_ode    = torch.mean(residual**2)

        # IC: theta(0) = pi/3
        loss_ic_theta = torch.mean((model(t_ic) - theta_ic_true)**2)

        # IC: dtheta/dt(0) = 0
        t_ic_grad = t_ic.clone().requires_grad_(True)
        theta_0   = model(t_ic_grad)
        omega_0   = torch.autograd.grad(
            theta_0, t_ic_grad,
            grad_outputs=torch.ones_like(theta_0),
            create_graph=True
        )[0]
        loss_ic_omega = torch.mean(omega_0**2)

        loss = loss_ode + 10.0 * loss_ic_theta + 10.0 * loss_ic_omega
        loss.backward()
        opt.step()

        history.append(loss.item())
        if epoch % 3000 == 0:
            print(f'  Epoch {epoch:6d} | Loss: {history[-1]:.3e} '
                  f'| ODE: {loss_ode.item():.3e} '
                  f'| IC_theta: {loss_ic_theta.item():.3e} '
                  f'| IC_omega: {loss_ic_omega.item():.3e}')

    return model, history


# Training call
pend_model, pend_history = train_pendulum_pinn(width=32, epochs=15000)
```

**Key points in the solution:**

1. The IC for the initial angle and the IC for the initial angular velocity are two separate loss terms. Both must be included; omitting $\dot{\theta}(0)=0$ is a common mistake that leads to incorrect oscillation phase.
2. IC weights of 10 are chosen because the PDE residual is averaged over 3 000 collocation points but each IC has only a single evaluation point. The higher weight compensates for this imbalance.
3. The `t_ic_grad` tensor used to compute $\omega(0)$ must have `requires_grad=True` independently of `t_ic`, because autograd needs a leaf node to differentiate through.
4. Collocation points are fixed during training (not resampled each epoch). Adaptive resampling (adding more points where the residual is large) would improve accuracy for long time windows — this is the topic of [Session 10](Session10.ipynb).

</details>

### 3.6 Discussion questions

Once your PINN is trained and producing reasonable results, consider the following questions. There are no single correct answers — they are prompts for reflection.

1. **Large initial angles.** Change `THETA0` to $\pi \times 0.9$ (just under 180°). Does the PINN still converge? What changes about the solution qualitatively, and why does this make the training harder? (Hint: consider how rapidly $\sin\theta$ changes near $\theta = \pi$.)

2. **Adding damping.** Modify the ODE to include linear damping:
   $$
   \frac{d^2\theta}{dt^2} + 2\zeta\omega_0\frac{d\theta}{dt} + \frac{g}{L}\sin\theta = 0
   $$
   with $\zeta = 0.1$ and $\omega_0 = \sqrt{g/L}$. The damping term $2\zeta\omega_0\,d\theta/dt$ is already available as the first derivative computed inside `pendulum_ode_residual`. Does the PINN adapt naturally, or does it require retuning of the loss weights?

3. **Inverse problem.** Suppose $g$ is unknown. Make it a learnable `nn.Parameter` initialised at $g = 5.0$, add a few noisy observations from `theta_ref` as a data loss term, and train jointly. Does the PINN recover $g \approx 9.81$?

4. **Longer time windows.** Extend `T_END` to 20 s. Does the PINN maintain accuracy throughout? What are the symptoms of failure, and which training strategies from [Sessions 9](Session9.ipynb)–[10](Session10.ipynb) might help?

---

## 4. Part 3: course summary

### 4.1 The 12-session course

The table below summarises the full course. Each row identifies the key concept introduced in that session.

| Session | Title | Key concept |
|:---:|---|---|
| [1](Session1.ipynb) | Foundations of differential equations in physics (Part 1) | PDEs in physics; explicit finite-difference (FD) schemes |
| [2](Session2.ipynb) | Foundations of differential equations in physics (Part 2) | Crank–Nicolson scheme; curse of dimensionality; limitations of grid-based methods |
| [3](Session3.ipynb) | Introduction to neural networks and deep learning (Part 1) | Multilayer perceptron (MLP); universal approximation theorem; backpropagation |
| [4](Session4.ipynb) | Automatic differentiation and neural network function approximation | PyTorch autograd; higher-order derivatives; function approximation with an MLP |
| [5](Session5.ipynb) | Introduction to physics-informed neural networks | The PINN loss function; collocation points; first PINN on the decay ODE |
| [6](Session6.ipynb) | PINN for the 1D heat equation | Parabolic PDE; second spatial derivative via autograd; comparison with exact solution |
| [7](Session7.ipynb) | PINN for Burgers' equation | Nonlinear advection-diffusion; near-shock behaviour; IC loss weighting |
| [8](Session8.ipynb) | Implementation in Python (Part 2) — inverse problems, loss balancing, and 2D Poisson | Inverse problems; learnable parameters; gradient-norm loss balancing; 2D spatial domain |
| [9](Session9.ipynb) | Tuning the physics loss weight $\lambda$ in PINNs | Systematic $\lambda$ sweep; interpolation vs extrapolation error; finding $\lambda^*$ |
| 10 | Advanced topics and extensions | Adaptive sampling; domain decomposition; spectral bias; self-adaptive weights |
| 11 | Applications — Schrödinger and Navier–Stokes equations | Quantum mechanics; fluid dynamics; complex-valued and vector-field PINNs |
| 12 | Applications and projects | Quantitative PINN vs FD comparison; nonlinear pendulum project; course synthesis |

### 4.2 Key strengths of PINNs

1. **Mesh-free formulation.** PINNs sample the domain randomly; there is no grid to construct. This removes the mesh-generation bottleneck for complex geometries and makes the method straightforward to implement in any dimension.
2. **Natural handling of inverse problems.** Unknown PDE parameters can be made learnable, allowing the physical model and the unknown quantities to be inferred simultaneously from data. Standard FD solvers require an outer optimisation loop for this.
3. **Unified treatment of data and physics.** Experimental observations enter as additional loss terms alongside the PDE residual. The network simultaneously satisfies the governing equations and fits the measurements, with the balance controlled by the loss weights.
4. **Differentiable continuous representation.** A trained PINN is a smooth, infinitely differentiable function that can be queried at any point in the domain without interpolation. Derived quantities (fluxes, gradients) are available at no extra implementation cost via autograd.
5. **Dimension scalability.** The number of collocation points grows only linearly with the dimension of the domain, in contrast to the exponential growth of grid points in FD or finite-element (FE) methods. PINNs are one of the few tractable approaches for PDEs in five or more dimensions.

### 4.3 Key limitations of PINNs

1. **Training cost relative to classical solvers.** On standard benchmark problems such as the 1D heat equation, even a modest Crank–Nicolson scheme outperforms a PINN by many orders of magnitude in wall-clock time at the same accuracy. Training a PINN is an iterative optimisation process; solving a tridiagonal system is not.
2. **Spectral bias.** Neural networks with smooth activations learn low-frequency components of the solution much faster than high-frequency ones. PINNs struggle with solutions that contain sharp gradients, shocks, or highly oscillatory behaviour unless specialised techniques (Fourier feature embeddings, adaptive activation functions, or domain decomposition) are applied.
3. **Loss balancing is non-trivial.** The total loss is a weighted sum of PDE, boundary condition, initial condition, and (optionally) data terms. The weights critically affect convergence and accuracy, but there is no universally correct procedure for setting them. Poor choices can cause one term to dominate and the others to be effectively ignored throughout training.
4. **Lack of convergence guarantees.** Gradient-descent training of neural networks is a non-convex optimisation. There is no guarantee of convergence to the global minimum, and the result depends on random seed, learning rate schedule, and network initialisation in ways that classical solvers do not.
5. **Verification and validation.** For a new problem, it is generally not possible to know a priori whether the PINN has converged to the correct solution. The PDE residual at collocation points is a necessary but not sufficient indicator of accuracy; the solution must be validated against a reference whenever possible.
6. **Current state of software maturity.** Established FD, finite-volume (FV), and finite-element libraries have decades of development, testing, and community use behind them. PINN software is comparatively immature, and best practices are still evolving rapidly.

### 4.4 Further reading

The following references are the most important starting points for deeper study.

**Foundational paper**

> Raissi, M., Perdikaris, P. & Karniadakis, G. E. (2019). Physics-informed neural networks: a deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, **378**, 686–707. https://doi.org/10.1016/j.jcp.2018.10.045

This is the paper that established the modern PINN framework. Sections 2–4 cover the loss formulation, forward problems, and inverse problems with the examples used throughout this course.

**Comprehensive review**

> Karniadakis, G. E., Kevrekidis, I. G., Lu, L., Perdikaris, P., Wang, S. & Yang, L. (2021). Physics-informed machine learning. *Nature Reviews Physics*, **3**, 422–440. https://doi.org/10.1038/s42254-021-00314-5

A broad survey of the field as of 2021, covering extensions beyond the original PINN framework including operator learning, uncertainty quantification, and hybrid methods. Suitable for a literature review or as background for a dissertation.

**Accessible tutorial**

> Moseley, B. (2023). *So, what is a physics-informed neural network?* https://benmoseley.blog/my-research/so-what-is-a-physics-informed-neural-network/

An approachable web tutorial with clean code examples. Highly recommended as a complement to this course for readers who prefer a more visual, less formal style.

**Loss balancing and failure modes**

> Wang, S., Yu, X. & Perdikaris, P. (2022). When and why PINNs fail to train: a neural tangent kernel perspective. *Journal of Computational Physics*, **449**, 110768. https://doi.org/10.1016/j.jcp.2021.110768

Explains theoretically why certain loss configurations cause training failure and derives NTK-based weight schemes to address them. The paper underlying the discussion in [Session 8](Session8.ipynb).

### 4.5 Congratulations

You have reached the end of the course. Over twelve sessions you have:

- Reviewed the classical theory of ODEs and PDEs and the finite-difference methods that underpin computational physics;
- Built an understanding of neural networks from first principles, including the universal approximation theorem, backpropagation, and automatic differentiation;
- Implemented the full PINN framework in PyTorch, solving forward and inverse problems across a range of physically significant equations — the heat equation, Burgers' equation, the Poisson equation, the Schrödinger equation, and the Navier–Stokes equations;
- Studied the practical challenges of PINN training: loss balancing, the choice of $\lambda$, spectral bias, and adaptive sampling;
- Carried out a quantitative, fair comparison between PINNs and classical finite-difference methods and articulated when each approach is preferable;
- Completed an independent project on the nonlinear pendulum, applying the full workflow to an equation you had not seen solved with a PINN before.

Physics-informed machine learning is a rapidly developing field. The techniques you have learned here are directly applicable to research problems in continuum mechanics, climate modelling, biomedical simulation, and beyond. The best way to consolidate what you have learned is to apply it to a problem from your own area of interest, ideally one where an exact or high-fidelity numerical reference is available for comparison.